Kumo-RFM Policy Notebook (VN2)

Note. Out-of-stock weeks are treated as censored (set to NaN). All statistics/features are computed on available weeks only; true zeros are kept. Future demand is modeled under full availability.

References:
- Hands-on RFM policy patterns [link](https://github.com/kumo-ai/kumo-rfm/blob/033b2f91899a83589184bffe466f078912e1350c/notebooks/hands_on.ipynb#L1072)
- Quickstart policy mapping [link](https://github.com/kumo-ai/kumo-rfm/blob/033b2f91899a83589184bffe466f078912e1350c/notebooks/quickstart.ipynb#L690)



In [ ]:
# Setup and paths
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA = Path("../data").resolve()
SUB = Path("../submissions").resolve(); SUB.mkdir(exist_ok=True)
ART = Path("../artifacts/rfm").resolve(); ART.mkdir(parents=True, exist_ok=True)

FILES = {
    "sales": DATA / "Week 0 - 2024-04-08 - Sales.csv",
    "in_stock": DATA / "Week 0 - In Stock.csv",
    "initial": DATA / "Week 0 - 2024-04-08 - Initial State.csv",
    "template": DATA / "Week 0 - Submission Template.csv",
}

INDEX = ["Store","Product"]



In [ ]:
# Availability-aware load and RFM features
sw = pd.read_csv(FILES["sales"]).set_index(INDEX)
iw = pd.read_csv(FILES["in_stock"]).set_index(INDEX)

sw.columns = pd.to_datetime(sw.columns)
iw.columns = pd.to_datetime(iw.columns)

avail = iw.astype(bool)
sales_c = sw.where(avail)

# RFM-like features (recency: last non-NaN week; frequency: nonzero weeks; monetary: mean positive)
weeks = sales_c.columns
last_week = weeks.max()

# Recency: weeks since last observed positive sale (on available weeks)
last_pos_week = sales_c.apply(lambda r: weeks[r.gt(0)].max() if r.gt(0).any() else pd.NaT, axis=1)
recency_weeks = (last_week - last_pos_week).dt.days.div(7).fillna(np.inf)

# Frequency: share of available weeks with positive sales
available_weeks = avail.sum(axis=1).clip(lower=1)
positive_weeks = (sales_c.fillna(0) > 0).sum(axis=1)
frequency = (positive_weeks / available_weeks).astype(float)

# Monetary: mean positive demand on available weeks
pos_counts = (sales_c > 0).sum(axis=1)
monetary = (sales_c.where(sales_c > 0).sum(axis=1) / pos_counts.replace(0, np.nan)).fillna(0.0)

rfm = pd.DataFrame({
    "recency_weeks": recency_weeks,
    "frequency": frequency,
    "monetary": monetary,
})
rfm.to_csv(ART / "rfm_features.csv")
rfm.head()



In [ ]:
# RFM -> policy mapping (inspired by Kumo quickstart)
# Map recency/frequency/monetary to a per-SKU k multiplier in [0.7, 1.3]

# Normalize features
r = np.log1p(rfm["recency_weeks"]).replace(np.inf, np.nan).fillna(rfm["recency_weeks"].max())
f = rfm["frequency"].clip(0,1)
m = np.log1p(rfm["monetary"]).fillna(0.0)

r_z = (r - r.median()) / (r.mad() + 1e-6)
f_z = (f - f.median()) / (f.mad() + 1e-6)
m_z = (m - m.median()) / (m.mad() + 1e-6)

# Heuristic linear map then squashed to [0.7, 1.3]
lin = -0.6*r_z + 0.8*f_z + 0.4*m_z
k0 = 1.0 + 0.2 * np.tanh(lin)

k0.describe(), k0.head()



In [ ]:
# InventorySim wiring and Optuna CV
import optuna
from vn2inventory.sim_env import InventorySim, Costs
from vn2inventory.policy import _inv_normal_cdf

# Baseline stats for μ, σ (availability-aware)
mu = sales_c.mean(axis=1, skipna=True).fillna(0.0)
sigma = sales_c.std(axis=1, ddof=1, skipna=True).fillna(mu.clip(lower=0.0).pow(0.5))

# Sim setup
sales_wide = pd.read_csv(FILES["sales"])  # wide with INDEX columns present
initial_state = pd.read_csv(FILES["initial"]) 
demand_dates = [c for c in sales_wide.columns if c not in INDEX]

sim = InventorySim(
    sales_wide=sales_wide,
    initial_state=initial_state,
    costs=Costs(holding_per_unit=0.2, shortage_per_unit=1.0),
    index_cols=("Store","Product"),
    demand_dates=demand_dates,
)

P = 3

# Order policy using per-SKU k (from RFM) and a global service level
k_series = k0.copy()

def eval_params(service_level: float, k_scale: float) -> float:
    sim.reset_to(initial_state)
    total = 0.0
    z = _inv_normal_cdf(service_level)
    k = (k_scale * k_series).clip(0.7, 1.3)
    for _ in demand_dates:
        inv_pos = sim.inventory_position()
        tgt = k.reindex(inv_pos.index).fillna(1.0) * (mu * P + z * sigma * np.sqrt(P))
        orders = (tgt - inv_pos).clip(lower=0.0).round().astype(int)
        info = sim.step(orders)
        total += info["round_cost"]
    return float(total)


def objective(trial: optuna.Trial) -> float:
    sl = trial.suggest_float("service_level", 0.88, 0.99)
    ks = trial.suggest_float("k_scale", 0.7, 1.3)
    return eval_params(sl, ks)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40, show_progress_bar=False)
best = study.best_params
best



In [ ]:
# Build submission with tuned params
sl = float(best.get("service_level", 0.95))
ks = float(best.get("k_scale", 1.0))

# One-step orders for Week 1
sim.reset_to(initial_state)
inv_pos_now = sim.inventory_position()
z = _inv_normal_cdf(sl)
k = (ks * k_series).clip(0.7, 1.3)
mu_s = mu.reindex(inv_pos_now.index).fillna(0.0)
sigma_s = sigma.reindex(inv_pos_now.index).fillna(mu_s.clip(lower=0.0).pow(0.5))

P = 3
target = k.reindex(inv_pos_now.index).fillna(1.0) * (mu_s * P + z * sigma_s * np.sqrt(P))
orders_now = (target - inv_pos_now).clip(lower=0.0).round().astype(int)

index_df = pd.read_csv(FILES["template"])[INDEX].set_index(INDEX)
submission = index_df.copy(); submission["0"] = orders_now.reindex(index_df.index).fillna(0).astype(int).values
out_csv = SUB / "orders_kumo_rfm.csv"
submission.reset_index().to_csv(out_csv, index=False)
print({"submission": str(out_csv), "rows": len(submission)})

